# Part 3 - Support Vector Machines (SVM)
**Gray Interface '26 | Task 3**

Dataset: [Dry Bean Dataset](https://www.kaggle.com/datasets/muratkokludataset/dry-bean-dataset)

## Importing the dependencies

In [ ]:
!pip install kagglehub openpyxl --quiet

In [ ]:
import kagglehub
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)


## Loading the Dataset

In [ ]:
path = kagglehub.dataset_download("muratkokludataset/dry-bean-dataset")
print(os.listdir(path))


In [ ]:
# Dataset comes as an Excel file
excel_file = [f for f in os.listdir(path) if f.endswith('.xlsx')][0]
df = pd.read_excel(os.path.join(path, excel_file))

print("Shape:", df.shape)
df.head()


## Exploratory Data Analysis

In [ ]:
print(df.info())
print("\nMissing values:", df.isnull().sum().sum())


In [ ]:
# Class distribution
plt.figure(figsize=(8, 4))
df['Class'].value_counts().plot(kind='bar', color='steelblue')
plt.title('Bean Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# Feature distributions
df.drop(columns=['Class']).hist(figsize=(14, 10), bins=30, color='steelblue')
plt.suptitle('Feature Distributions', y=1.01)
plt.tight_layout()
plt.show()


## Preprocessing

In [ ]:
# Encode class labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['Class'])
print("Classes:", list(le.classes_))

X = df.drop(columns=['Class', 'label'])
y = df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features — SVM is very sensitive to feature magnitude
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")


## Training SVM with Different Kernels

In [ ]:
# Train each kernel and record time and metrics
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
kernel_results = []

for kernel in kernels:
    print(f"Training {kernel} kernel...", end=' ')
    start = time.time()

    # Use default C=1, degree=3 for poly
    model = SVC(kernel=kernel, C=1.0, random_state=42, probability=True)
    model.fit(X_train_scaled, y_train)
    elapsed = round(time.time() - start, 2)

    preds = model.predict(X_test_scaled)
    kernel_results.append({
        'Kernel':    kernel,
        'Train Time (s)': elapsed,
        'Accuracy':  round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds, average='weighted'), 4),
        'Recall':    round(recall_score(y_test, preds, average='weighted'), 4),
        'F1':        round(f1_score(y_test, preds, average='weighted'), 4),
    })
    print(f"done in {elapsed}s")

kernel_df = pd.DataFrame(kernel_results)
print("\n", kernel_df.to_string(index=False))


## Comparing Kernels — Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

kernel_df.plot(kind='bar', x='Kernel', y='F1', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('F1 Score by Kernel')
axes[0].set_ylabel('F1 Score')
axes[0].set_xticklabels(kernel_df['Kernel'], rotation=0)
axes[0].set_ylim(0, 1)

kernel_df.plot(kind='bar', x='Kernel', y='Train Time (s)', ax=axes[1], color='#E50914', legend=False)
axes[1].set_title('Training Time by Kernel')
axes[1].set_ylabel('Time (seconds)')
axes[1].set_xticklabels(kernel_df['Kernel'], rotation=0)

plt.tight_layout()
plt.show()


## Experimenting with C and Gamma (RBF Kernel)

In [ ]:
# C controls how much we penalize misclassifications
# Gamma controls how far the influence of a single training point reaches

C_values     = [0.1, 1.0, 10.0, 100.0]
gamma_values = ['scale', 'auto', 0.001, 0.01]

c_gamma_results = []
for C in C_values:
    for gamma in gamma_values:
        model = SVC(kernel='rbf', C=C, gamma=gamma, random_state=42)
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        c_gamma_results.append({
            'C': C, 'Gamma': str(gamma),
            'F1': round(f1_score(y_test, preds, average='weighted'), 4),
            'Accuracy': round(accuracy_score(y_test, preds), 4),
        })

cg_df = pd.DataFrame(c_gamma_results)

# Pivot for heatmap
pivot = cg_df.pivot(index='C', columns='Gamma', values='F1')
plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd')
plt.title('F1 Score — RBF Kernel (C vs Gamma)')
plt.tight_layout()
plt.show()


## Polynomial Kernel — Effect of Degree

In [ ]:
degree_results = []
for degree in [2, 3, 4, 5]:
    model = SVC(kernel='poly', degree=degree, C=1.0, random_state=42)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    degree_results.append({
        'Degree':   degree,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'F1':       round(f1_score(y_test, preds, average='weighted'), 4),
    })

deg_df = pd.DataFrame(degree_results)
print(deg_df.to_string(index=False))

plt.figure(figsize=(7, 4))
plt.plot(deg_df['Degree'], deg_df['F1'], marker='o', color='steelblue')
plt.title('Polynomial Kernel — F1 vs Degree')
plt.xlabel('Degree')
plt.ylabel('F1 Score')
plt.tight_layout()
plt.show()


## Best Model — Confusion Matrix

In [ ]:
# Use best kernel (likely RBF) with best C and gamma from the heatmap
best_model = SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42)
best_model.fit(X_train_scaled, y_train)
preds = best_model.predict(X_test_scaled)

print(classification_report(y_test, preds, target_names=le.classes_))

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — Best SVM (RBF, C=10, gamma=scale)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


## Observations

### Kernel Comparison
- **Linear kernel**: Fast to train, works well when classes are linearly separable. Dry beans have overlapping physical features so linear does okay but not great.
- **RBF kernel**: Best overall — maps data into higher dimensions where it becomes more separable. Works well for most real-world datasets.
- **Polynomial kernel**: Competitive but slower as degree increases. Degree=3 is usually the sweet spot.
- **Sigmoid kernel**: Often underperforms for multi-class problems — less stable than RBF.

### Effect of C
- Low C (0.1): Large margin, more misclassifications allowed — may underfit.
- High C (100): Tiny margin, tries to classify everything correctly — risk of overfitting.
- C=10 with RBF gave the best balance on this dataset.

### Effect of Gamma (RBF)
- High gamma: Each training point influences only a very small region — can overfit.
- Low gamma: Smoother decision boundary — can underfit.
- `gamma='scale'` (1 / n_features × variance) is usually a safe default and performed well here.

### Feature Scaling
SVM depends entirely on distances — without StandardScaler, features like `Area` (large values) would completely dominate `Compactness` (small values), making the kernel computation meaningless.
